## County population: prepare & merge ACS year files (2000–2015 subset)

This notebook loads county-level population metadata (2000–2010), then appends ACS population estimates for selected years and writes cleaned CSV(s).

### Inputs
- project_root = .. (notebook root)
- Raw metadata: 00_data/raw/county_population/Population_2000-2010.csv (meta table)
- Raw ACS files: 00_data/raw/county_population/Population_{year}.csv (year = 2011, 2012, …)

### Key variables created in the first code cell
- meta_path, acs_path, output_path — pathlib.PosixPath for the left (metadata), one ACS year, and per-year output file.

### What the notebook does
1. Read metadata CSV (meta_df).
2. Clean metadata:
    - drop POPESTIMATE2000 if present,
    - keep only county rows (SUMLEV == 50),
    - build zero-padded STATE_FIPS, COUNTY_FIPS and FIPS.
3. Define helper load_acs_population_for_year(year):
    - constructs path for Population_{year}.csv,
    - reads CSV with latin1 + ignore errors,
    - strips BOM/quotes from column names,
    - removes header-like rows ("Geography"),
    - derives FIPS from GEO_ID and renames S0101_C01_001E → POPESTIMATE{year},
    - returns DataFrame with FIPS, GEO_ID, NAME, POPESTIMATE{year}.
4. Loop over years (years list) and inner-join each year's POPESTIMATE on meta_df by FIPS.
5. Remove Alaska rows (STNAME == "Alaska") from merged result.
6. Save merged table(s) to clean CSV(s) under 00_data/clean.

### Outputs
- Full merged output_all_path (county_population_2000_2015.csv)

## Input

In [98]:
import pandas as pd
from pathlib import Path

# Notebook is located under 10_code
project_root = Path("..")

# Left file 2000-2010
meta_path = (
    project_root / "00_data" / "raw" / "county_population" / "Population_2000-2010.csv"
)

# Right file 2011 (testing the path is right or not)
year = 2011
acs_path = (
    project_root / "00_data" / "raw" / "county_population" / f"Population_{year}.csv"
)

# Output file path (can be renamed as needed)
output_path = project_root / "00_data" / "clean" / f"county_population_2000_{year}.csv"

meta_path, acs_path, output_path

(PosixPath('../00_data/raw/county_population/Population_2000-2010.csv'),
 PosixPath('../00_data/raw/county_population/Population_2011.csv'),
 PosixPath('../00_data/clean/county_population_2000_2011.csv'))

## Data Preperation - 2000-2010 population data

In [99]:
# Read metadata file with county info (SUMLEV, STATE, COUNTY, STNAME, CTYNAME, ...)
meta_df = pd.read_csv(meta_path, encoding="latin1")

print("Original left table shape:", meta_df.shape)
meta_df.head()

Original left table shape: (3194, 20)


,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,ESTIMATESBASE2000,POPESTIMATE2000,POPESTIMATE2001,POPESTIMATE2002,POPESTIMATE2003,POPESTIMATE2004,POPESTIMATE2005,POPESTIMATE2006,POPESTIMATE2007,POPESTIMATE2008,POPESTIMATE2009,CENSUS2010POP,POPESTIMATE2010
0,40,3,6,1,0,Alabama,Alabama,4447207,4452173,4467634,4480089,4503491,4530729,4569805,4628981,4672840,4718206,4757938,4779736,4785298
1,50,3,6,1,1,Alabama,Autauga County,43751,44021,44889,45909,46800,48366,49676,51328,52405,53277,54135,54571,54632
2,50,3,6,1,3,Alabama,Baldwin County,140416,141342,144875,147957,151509,156266,162183,168121,172404,175827,179406,182265,183195
3,50,3,6,1,5,Alabama,Barbour County,29042,29015,28863,28653,28594,28287,28027,27861,27757,27808,27657,27457,27411
4,50,3,6,1,7,Alabama,Bibb County,19856,19913,21028,21199,21399,21721,22042,22099,22438,22705,22941,22915,22867


In [100]:
# Drop the only unwanted column POPESTIMATE2000
if "POPESTIMATE2000" in meta_df.columns:
    meta_df = meta_df.drop(columns=["POPESTIMATE2000"])

# Keep only county-level rows (SUMLEV == 50)
meta_df = meta_df.query("SUMLEV == 50").copy()

# Build FIPS (2-digit state + 3-digit county)
meta_df["STATE_FIPS"] = meta_df["STATE"].astype(int).astype(str).str.zfill(2)
meta_df["COUNTY_FIPS"] = meta_df["COUNTY"].astype(int).astype(str).str.zfill(3)
meta_df["FIPS"] = meta_df["STATE_FIPS"] + meta_df["COUNTY_FIPS"]

print("Cleaned left table shape:", meta_df.shape)
meta_df.head()

Cleaned left table shape: (3143, 22)


,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,ESTIMATESBASE2000,POPESTIMATE2001,POPESTIMATE2002,...,POPESTIMATE2005,POPESTIMATE2006,POPESTIMATE2007,POPESTIMATE2008,POPESTIMATE2009,CENSUS2010POP,POPESTIMATE2010,STATE_FIPS,COUNTY_FIPS,FIPS
1,50,3,6,1,1,Alabama,Autauga County,43751,44889,45909,...,49676,51328,52405,53277,54135,54571,54632,01,001,01001
2,50,3,6,1,3,Alabama,Baldwin County,140416,144875,147957,...,162183,168121,172404,175827,179406,182265,183195,01,003,01003
3,50,3,6,1,5,Alabama,Barbour County,29042,28863,28653,...,28027,27861,27757,27808,27657,27457,27411,01,005,01005
4,50,3,6,1,7,Alabama,Bibb County,19856,21028,21199,...,22042,22099,22438,22705,22941,22915,22867,01,007,01007
5,50,3,6,1,9,Alabama,Blount County,50982,51845,52551,...,54624,55485,56240,57055,57341,57322,57338,01,009,01009


## Data Preperation - 2011-2015 population data and merge 2000-2010

In [101]:
def load_acs_population_for_year(year: int) -> pd.DataFrame:
    acs_path = (
        project_root
        / "00_data"
        / "raw"
        / "county_population"
        / f"Population_{year}.csv"
    )
    print(f"Loading: {acs_path}")

    # Load with BOM-safe settings
    acs_df = pd.read_csv(acs_path, encoding="latin1", encoding_errors="ignore")

    # Fix column names (BOM + quotes)
    acs_df.columns = acs_df.columns.str.replace("ï»¿", "", regex=False).str.replace(
        '"', "", regex=False
    )

    # Remove header-like row
    acs_df = acs_df[acs_df["GEO_ID"] != "Geography"].copy()

    # Build FIPS
    acs_df["FIPS"] = acs_df["GEO_ID"].astype(str).str[-5:]

    # Rename S0101_C01_001E to POPESTIMATE{year}
    pop_col = f"POPESTIMATE{year}"
    acs_df = acs_df[["FIPS", "GEO_ID", "NAME", "S0101_C01_001E"]].rename(
        columns={"S0101_C01_001E": pop_col}
    )

    return acs_df

In [102]:
years = [2011, 2012, 2013, 2014, 2015]

merged_all = meta_df.copy()

for y in years:
    acs_y = load_acs_population_for_year(y)

    # Keep only FIPS and POPESTIMATE{year} columns (GEO_ID/NAME only needed for first iteration)
    pop_col = f"POPESTIMATE{y}"
    acs_y = acs_y[["FIPS", pop_col]]

    merged_all = merged_all.merge(acs_y, on="FIPS", how="inner")

# Remove Alaska from the dataset
merged_all = merged_all.loc[merged_all["STNAME"] != "Alaska"]

# Add state abbreviations (STATE_ABBR)

state_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
}

# Add abbreviation column
merged_all["STATE_ABBR"] = merged_all["STNAME"].map(state_abbrev)

# Check
merged_all[["STNAME", "STATE_ABBR"]].drop_duplicates().sort_values("STNAME")

merged_all.head()

Loading: ../00_data/raw/county_population/Population_2011.csv
Loading: ../00_data/raw/county_population/Population_2012.csv
Loading: ../00_data/raw/county_population/Population_2013.csv
Loading: ../00_data/raw/county_population/Population_2014.csv
Loading: ../00_data/raw/county_population/Population_2015.csv


,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,ESTIMATESBASE2000,POPESTIMATE2001,POPESTIMATE2002,...,POPESTIMATE2010,STATE_FIPS,COUNTY_FIPS,FIPS,POPESTIMATE2011,POPESTIMATE2012,POPESTIMATE2013,POPESTIMATE2014,POPESTIMATE2015,STATE_ABBR
0,50,3,6,1,1,Alabama,Autauga County,43751,44889,45909,...,54632,01,001,01001,53944,54590,54907,55136,55221,AL
1,50,3,6,1,3,Alabama,Baldwin County,140416,144875,147957,...,183195,01,003,01003,179523,183226,187114,191205,195121,AL
2,50,3,6,1,5,Alabama,Barbour County,29042,28863,28653,...,27411,01,005,01005,27546,27469,27321,27119,26932,AL
3,50,3,6,1,7,Alabama,Bibb County,19856,21028,21199,...,22867,01,007,01007,22746,22769,22754,22653,22604,AL
4,50,3,6,1,9,Alabama,Blount County,50982,51845,52551,...,57338,01,009,01009,57140,57466,57623,57645,57710,AL


## Basic health check

In [103]:
# ================================
# BASIC HEALTH CHECK FOR merged_all
# ================================

print("▶ Dataset shape:", merged_all.shape)
print("\n")

# 1. Check nulls for all columns
print("▶ NULL values per column:")
display(merged_all.isnull().sum())
print("\n")

# 2. Check if STNAME has missing values
print("▶ Null count for STNAME:", merged_all["STNAME"].isnull().sum())
print("\n")

# 3. Check if FIPS has missing values
print("▶ Null count for FIPS:", merged_all["FIPS"].isnull().sum())
print("\n")

# 4. Check duplicated FIPS (should be 0)
print("▶ Number of duplicated FIPS:", merged_all["FIPS"].duplicated().sum())
print("\n")

# 5. Check nulls in all population columns
pop_cols = [col for col in merged_all.columns if col.startswith("POPESTIMATE")]
print("▶ Population columns detected:", pop_cols)
print("\n▶ NULL check for population columns:")
display(merged_all[pop_cols].isnull().sum())
print("\n")

# 6. List unique STNAME values
print("▶ Unique STNAME list:")
display(sorted(merged_all["STNAME"].unique()))
display(merged_all["STNAME"].nunique())
print("\n")

# 7. Count counties per state (STNAME)
print("▶ County count by state (STNAME):")
display(merged_all["STNAME"].value_counts())
print("\n")

# 8. Check dtypes of population columns (should be numeric)
print("▶ Data types of population columns:")
display(merged_all[pop_cols].dtypes)
print("\n")

# 9. Quick sanity check — sum population for a sample state (Florida)
if "POPESTIMATE2010" in merged_all.columns:
    fl_pop_2010 = merged_all.loc[
        merged_all["STNAME"] == "Florida", "POPESTIMATE2010"
    ].sum()
    print("▶ Florida 2010 total population (sanity check):", fl_pop_2010)
else:
    print("▶ Skipped Florida sanity check because POPESTIMATE2010 not found")

▶ Dataset shape: (3112, 28)


▶ NULL values per column:


SUMLEV               0
REGION               0
DIVISION             0
STATE                0
COUNTY               0
STNAME               0
CTYNAME              0
ESTIMATESBASE2000    0
POPESTIMATE2001      0
POPESTIMATE2002      0
POPESTIMATE2003      0
POPESTIMATE2004      0
POPESTIMATE2005      0
POPESTIMATE2006      0
POPESTIMATE2007      0
POPESTIMATE2008      0
POPESTIMATE2009      0
CENSUS2010POP        0
POPESTIMATE2010      0
STATE_FIPS           0
COUNTY_FIPS          0
FIPS                 0
POPESTIMATE2011      0
POPESTIMATE2012      0
POPESTIMATE2013      0
POPESTIMATE2014      0
POPESTIMATE2015      0
STATE_ABBR           0
dtype: int64



▶ Null count for STNAME: 0


▶ Null count for FIPS: 0


▶ Number of duplicated FIPS: 0


▶ Population columns detected: ['POPESTIMATE2001', 'POPESTIMATE2002', 'POPESTIMATE2003', 'POPESTIMATE2004', 'POPESTIMATE2005', 'POPESTIMATE2006', 'POPESTIMATE2007', 'POPESTIMATE2008', 'POPESTIMATE2009', 'POPESTIMATE2010', 'POPESTIMATE2011', 'POPESTIMATE2012', 'POPESTIMATE2013', 'POPESTIMATE2014', 'POPESTIMATE2015']

▶ NULL check for population columns:


POPESTIMATE2001    0
POPESTIMATE2002    0
POPESTIMATE2003    0
POPESTIMATE2004    0
POPESTIMATE2005    0
POPESTIMATE2006    0
POPESTIMATE2007    0
POPESTIMATE2008    0
POPESTIMATE2009    0
POPESTIMATE2010    0
POPESTIMATE2011    0
POPESTIMATE2012    0
POPESTIMATE2013    0
POPESTIMATE2014    0
POPESTIMATE2015    0
dtype: int64



▶ Unique STNAME list:


...



▶ County count by state (STNAME):


STNAME
Texas                   254
Georgia                 159
Virginia                133
Kentucky                120
Missouri                115
Kansas                  105
Illinois                102
North Carolina          100
Iowa                     99
Tennessee                95
Nebraska                 93
Indiana                  92
Ohio                     88
Minnesota                87
Michigan                 83
Mississippi              82
Oklahoma                 77
Arkansas                 75
Wisconsin                72
Florida                  67
Pennsylvania             67
Alabama                  67
South Dakota             65
Colorado                 64
Louisiana                64
New York                 62
California               58
Montana                  56
West Virginia            55
North Dakota             53
South Carolina           46
Idaho                    44
Washington               39
Oregon                   36
New Mexico               33
Utah         



▶ Data types of population columns:


POPESTIMATE2001     int64
POPESTIMATE2002     int64
POPESTIMATE2003     int64
POPESTIMATE2004     int64
POPESTIMATE2005     int64
POPESTIMATE2006     int64
POPESTIMATE2007     int64
POPESTIMATE2008     int64
POPESTIMATE2009     int64
POPESTIMATE2010     int64
POPESTIMATE2011    object
POPESTIMATE2012    object
POPESTIMATE2013    object
POPESTIMATE2014    object
POPESTIMATE2015    object
dtype: object



▶ Florida 2010 total population (sanity check): 18843326


## Output

In [104]:
output_all_path = project_root / "00_data" / "clean" / "county_population_2000_2015.csv"
merged_all.to_csv(output_all_path, index=False)
output_all_path

PosixPath('../00_data/clean/county_population_2000_2015.csv')